In [23]:
# import/path設定

from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    f1_score,
)


# ============================================================
# Repository
# ============================================================

AI_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI"
)

if str(AI_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(AI_ROOT),
    )


from src.dataset import (
    load_normalization_stats,
    load_split_manifest,
    select_split,
    make_dataset,
)


# ============================================================
# Dataset
# ============================================================

DATASET_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset"
)

CLIP_ROOT = (
    DATASET_ROOT
    / "processed"
    / "clips_16k_1s"
)

AI05_META = (
    DATASET_ROOT
    / "metadata"
    / "05_split_info"
)

SPLIT_MANIFEST_PATH = (
    AI05_META
    / "split_manifest.csv"
)

NORMALIZATION_PATH = (
    AI05_META
    / "normalization_stats.npz"
)


# ============================================================
# Model
# ============================================================

FLOAT32_MODEL_PATH = (
    AI_ROOT
    / "model"
    / "baseline_float32.keras"
)

INT8_MODEL_PATH = (
    AI_ROOT
    / "model"
    / "baseline_int8.tflite"
)


# ============================================================
# AI-07 result
# ============================================================

AI07_META = (
    DATASET_ROOT
    / "metadata"
    / "07_evaluation_info"
)

FLOAT32_PREDICTION_PATH = (
    AI07_META
    / "test_predictions.csv"
)


# ============================================================
# AI-08 output
# ============================================================

AI08_META = (
    DATASET_ROOT
    / "metadata"
    / "08_quant_info"
)

AI08_META.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Constants
# ============================================================

CLASS_NAMES = [
    "OTHER",
    "SNORE",
    "COUGH",
]

CLASS_IDS = [
    0,
    1,
    2,
]


print("TensorFlow:", tf.__version__)
print("AI_ROOT:", AI_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)

TensorFlow: 2.20.0
AI_ROOT: /content/drive/MyDrive/TRON/AI
DATASET_ROOT: /content/drive/MyDrive/TRON/AI/dataset


In [2]:
# manifest/normalization/TRAIN/TEST の読み込み

manifest = load_split_manifest(
    SPLIT_MANIFEST_PATH
)

mean, std = load_normalization_stats(
    NORMALIZATION_PATH
)

train_df = select_split(
    manifest,
    "TRAIN",
)

test_df = select_split(
    manifest,
    "TEST",
)


print("TRAIN:", len(train_df))
print("TEST :", len(test_df))

print()
print("TRAIN class distribution:")
print(
    train_df["class_name"]
    .value_counts()
)

print()
print("TEST class distribution:")
print(
    test_df["class_name"]
    .value_counts()
)


assert len(train_df) == 5425
assert len(test_df) == 1162

assert mean.shape == (40,)
assert std.shape == (40,)

TRAIN: 5425
TEST : 1162

TRAIN class distribution:
class_name
COUGH    3586
OTHER    1064
SNORE     775
Name: count, dtype: int64

TEST class distribution:
class_name
COUGH    768
OTHER    228
SNORE    166
Name: count, dtype: int64


In [3]:
# float32 model 確認

float32_model = (
    tf.keras.models.load_model(
        FLOAT32_MODEL_PATH
    )
)

float32_model.summary()

print()
print(
    "Input shape :",
    float32_model.input_shape
)

print(
    "Output shape:",
    float32_model.output_shape
)

print(
    "Parameters  :",
    float32_model.count_params()
)


assert float32_model.input_shape == (
    None,
    98,
    40,
    1,
)

assert float32_model.output_shape == (
    None,
    3,
)

assert float32_model.count_params() == 6155

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 98, 40, 8)      │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 98, 40, 8)      │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 98, 40, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 49, 20, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 49, 20, 16)     │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 49, 20, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 49, 20, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 24, 10, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 10, 32)     │         4,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 24, 10, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 24, 10, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,243 (71.27 KB)

 Trainable params: 6,043 (23.61 KB)

 Non-trainable params: 112 (448.00 B)

 Optimizer params: 12,088 (47.22 KB)


Input shape : (None, 98, 40, 1)
Output shape: (None, 3)
Parameters  : 6155


In [4]:
# Reprasentative Dataset 作成

representative_ds = make_dataset(
    train_df,
    clip_root=CLIP_ROOT,
    mean=mean,
    std=std,
    training=False,
    batch_size=1,
    cache=False,
)


def representative_dataset():
    """
    Full-int8 PTQ calibration dataset.

    TRAIN split only.
    Input:
        float32 [1, 98, 40, 1]
    """

    count = 0

    for feature, _ in representative_ds:

        assert feature.shape == (
            1,
            98,
            40,
            1,
        )

        assert feature.dtype == tf.float32

        yield [feature]

        count += 1

    if count != len(train_df):
        raise RuntimeError(
            f"Representative sample count mismatch: "
            f"{count} != {len(train_df)}"
        )

In [5]:
sample_feature, sample_label = next(
    iter(representative_ds)
)

print(
    "Feature shape:",
    sample_feature.shape
)

print(
    "Feature dtype:",
    sample_feature.dtype
)

print(
    "Label:",
    sample_label.numpy()
)

print(
    "Feature min:",
    float(
        tf.reduce_min(
            sample_feature
        )
    )
)

print(
    "Feature max:",
    float(
        tf.reduce_max(
            sample_feature
        )
    )
)

Feature shape: (1, 98, 40, 1)
Feature dtype: <dtype: 'float32'>
Label: [0]
Feature min: 0.4672809839248657
Feature max: 2.136122226715088


In [6]:
# Full-integer int8 PTQ

converter = (
    tf.lite.TFLiteConverter
    .from_keras_model(
        float32_model
    )
)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

converter.representative_dataset = (
    representative_dataset
)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = (
    tf.int8
)

converter.inference_output_type = (
    tf.int8
)


print(
    "Starting full-integer int8 conversion..."
)

int8_tflite_model = (
    converter.convert()
)

print(
    "Conversion completed"
)

print(
    "TFLite bytes:",
    len(int8_tflite_model)
)

Starting full-integer int8 conversion...
Saved artifact at '/tmp/tmp2pyqbe8q'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 98, 40, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  139865006024144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139865006024336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139865006025296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139865006022800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139865006024912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139865006026256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139865006026832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139865006026640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139865006025872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139865006027024: TensorSpec(shape=(), dtype=tf

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Conversion completed
TFLite bytes: 11616


In [7]:
# restore int8 model

INT8_MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with open(
    INT8_MODEL_PATH,
    "wb",
) as f:
    f.write(
        int8_tflite_model
    )


print(
    "Saved:",
    INT8_MODEL_PATH
)

print(
    "File size:",
    INT8_MODEL_PATH.stat().st_size,
    "bytes"
)

Saved: /content/drive/MyDrive/TRON/AI/model/baseline_int8.tflite
File size: 11616 bytes


In [8]:
interpreter = tf.lite.Interpreter(
    model_path=str(
        INT8_MODEL_PATH
    )
)

interpreter.allocate_tensors()


input_details = (
    interpreter.get_input_details()
)

output_details = (
    interpreter.get_output_details()
)


print("=== Input ===")
for k, v in input_details[0].items():
    if k in {
        "name",
        "shape",
        "dtype",
        "quantization",
        "quantization_parameters",
    }:
        print(k, ":", v)


print()
print("=== Output ===")
for k, v in output_details[0].items():
    if k in {
        "name",
        "shape",
        "dtype",
        "quantization",
        "quantization_parameters",
    }:
        print(k, ":", v)

=== Input ===
name : serving_default_input_layer:0
shape : [ 1 98 40  1]
dtype : <class 'numpy.int8'>
quantization : (0.021447716280817986, 2)
quantization_parameters : {'scales': array([0.02144772], dtype=float32), 'zero_points': array([2], dtype=int32), 'quantized_dimension': 0}

=== Output ===
name : StatefulPartitionedCall_1:0
shape : [1 3]
dtype : <class 'numpy.int8'>
quantization : (0.00390625, -128)
quantization_parameters : {'scales': array([0.00390625], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0}


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [9]:
input_info = input_details[0]
output_info = output_details[0]


assert input_info["dtype"] == np.int8
assert output_info["dtype"] == np.int8


input_scale, input_zero_point = (
    input_info["quantization"]
)

output_scale, output_zero_point = (
    output_info["quantization"]
)


print()
print(
    "Input scale     :",
    input_scale
)

print(
    "Input zero point:",
    input_zero_point
)

print(
    "Output scale     :",
    output_scale
)

print(
    "Output zero point:",
    output_zero_point
)


assert input_scale > 0
assert output_scale > 0


Input scale     : 0.021447716280817986
Input zero point: 2
Output scale     : 0.00390625
Output zero point: -128


In [10]:
# confirm model size

float32_file_size = (
    FLOAT32_MODEL_PATH
    .stat()
    .st_size
)

int8_file_size = (
    INT8_MODEL_PATH
    .stat()
    .st_size
)


print(
    "float32 .keras:",
    float32_file_size,
    "bytes"
)

print(
    "int8 .tflite  :",
    int8_file_size,
    "bytes"
)

print(
    "File size ratio:",
    int8_file_size
    / float32_file_size
)


float32_parameter_bytes = (
    float32_model.count_params()
    * 4
)

int8_parameter_bytes_theoretical = (
    float32_model.count_params()
)


print()
print(
    "Theoretical float32 parameter bytes:",
    float32_parameter_bytes
)

print(
    "Approx. int8 parameter bytes:",
    int8_parameter_bytes_theoretical
)

float32 .keras: 132633 bytes
int8 .tflite  : 11616 bytes
File size ratio: 0.08758001402365927

Theoretical float32 parameter bytes: 24620
Approx. int8 parameter bytes: 6155


In [11]:
test_ds = make_dataset(
    test_df,
    clip_root=CLIP_ROOT,
    mean=mean,
    std=std,
    training=False,
    batch_size=1,
    cache=False,
)


sample_x, sample_y = next(
    iter(test_ds)
)

print(
    sample_x.shape,
    sample_x.dtype
)

print(
    sample_y.shape,
    sample_y.dtype
)


assert sample_x.shape == (
    1,
    98,
    40,
    1,
)

assert sample_x.dtype == tf.float32

(1, 98, 40, 1) <dtype: 'float32'>
(1,) <dtype: 'int32'>


In [12]:
def quantize_int8(
    x: np.ndarray,
    scale: float,
    zero_point: int,
) -> np.ndarray:
    """
    float32 -> int8

    q = round(x / scale) + zero_point
    """

    if scale <= 0:
        raise ValueError(
            "Quantization scale must be positive"
        )

    q = np.round(
        x / scale
        + zero_point
    )

    q = np.clip(
        q,
        -128,
        127,
    )

    return q.astype(
        np.int8
    )


def dequantize_int8(
    q: np.ndarray,
    scale: float,
    zero_point: int,
) -> np.ndarray:
    """
    int8 -> float32

    x = (q - zero_point) * scale
    """

    return (
        (
            q.astype(np.float32)
            - zero_point
        )
        * scale
    )

In [13]:
x_float, y = next(
    iter(test_ds)
)

x_float = x_float.numpy()


x_int8 = quantize_int8(
    x_float,
    input_scale,
    input_zero_point,
)


print(
    "float shape:",
    x_float.shape
)

print(
    "int8 shape :",
    x_int8.shape
)

print(
    "int8 dtype :",
    x_int8.dtype
)

print(
    "int8 range :",
    x_int8.min(),
    x_int8.max()
)


interpreter.set_tensor(
    input_info["index"],
    x_int8,
)

interpreter.invoke()


y_int8 = interpreter.get_tensor(
    output_info["index"]
)

y_float = dequantize_int8(
    y_int8,
    output_scale,
    output_zero_point,
)


print()
print(
    "Raw int8 output:",
    y_int8
)

print(
    "Dequantized output:",
    y_float
)

print(
    "True class:",
    int(y.numpy()[0])
)

print(
    "Predicted class:",
    int(
        np.argmax(
            y_float[0]
        )
    )
)

float shape: (1, 98, 40, 1)
int8 shape : (1, 98, 40, 1)
int8 dtype : int8
int8 range : -34 86

Raw int8 output: [[ 118 -119 -127]]
Dequantized output: [[0.9609375  0.03515625 0.00390625]]
True class: 0
Predicted class: 0


In [14]:
y_true = []
y_pred_int8 = []
y_prob_int8 = []


for feature, label in test_ds:

    x_float = feature.numpy()

    x_int8 = quantize_int8(
        x_float,
        input_scale,
        input_zero_point,
    )

    interpreter.set_tensor(
        input_info["index"],
        x_int8,
    )

    interpreter.invoke()

    output_int8 = (
        interpreter.get_tensor(
            output_info["index"]
        )
    )

    output_float = (
        dequantize_int8(
            output_int8,
            output_scale,
            output_zero_point,
        )
    )

    predicted_class = int(
        np.argmax(
            output_float[0]
        )
    )

    y_true.append(
        int(
            label.numpy()[0]
        )
    )

    y_pred_int8.append(
        predicted_class
    )

    y_prob_int8.append(
        output_float[0]
    )


y_true = np.asarray(
    y_true,
    dtype=np.int32,
)

y_pred_int8 = np.asarray(
    y_pred_int8,
    dtype=np.int32,
)

y_prob_int8 = np.asarray(
    y_prob_int8,
    dtype=np.float32,
)


print(
    "y_true:",
    y_true.shape
)

print(
    "y_pred:",
    y_pred_int8.shape
)

print(
    "y_prob:",
    y_prob_int8.shape
)


assert y_true.shape == (1162,)
assert y_pred_int8.shape == (1162,)
assert y_prob_int8.shape == (
    1162,
    3,
)

assert np.isfinite(
    y_prob_int8
).all()

y_true: (1162,)
y_pred: (1162,)
y_prob: (1162, 3)


In [15]:
int8_accuracy = accuracy_score(
    y_true,
    y_pred_int8,
)

int8_macro_f1 = f1_score(
    y_true,
    y_pred_int8,
    average="macro",
)

int8_weighted_f1 = f1_score(
    y_true,
    y_pred_int8,
    average="weighted",
)


print(
    "INT8 TEST accuracy:",
    int8_accuracy
)

print(
    "INT8 Macro F1:",
    int8_macro_f1
)

print(
    "INT8 Weighted F1:",
    int8_weighted_f1
)

INT8 TEST accuracy: 0.9492254733218589
INT8 Macro F1: 0.9224896915312325
INT8 Weighted F1: 0.9495184113257241


In [16]:
cm_int8 = confusion_matrix(
    y_true,
    y_pred_int8,
    labels=CLASS_IDS,
)

print(
    "INT8 confusion matrix:"
)

print(
    cm_int8
)

INT8 confusion matrix:
[[206  13   9]
 [ 13 149   4]
 [ 18   2 748]]


In [17]:
report_int8 = classification_report(
    y_true,
    y_pred_int8,
    labels=CLASS_IDS,
    target_names=CLASS_NAMES,
    digits=4,
)

print(
    report_int8
)

              precision    recall  f1-score   support

       OTHER     0.8692    0.9035    0.8860       228
       SNORE     0.9085    0.8976    0.9030       166
       COUGH     0.9829    0.9740    0.9784       768

    accuracy                         0.9492      1162
   macro avg     0.9202    0.9250    0.9225      1162
weighted avg     0.9500    0.9492    0.9495      1162



In [18]:
(
    precision,
    recall,
    f1,
    support,
) = precision_recall_fscore_support(
    y_true,
    y_pred_int8,
    labels=CLASS_IDS,
    zero_division=0,
)


int8_metrics_df = pd.DataFrame({
    "class_id": CLASS_IDS,
    "class_name": CLASS_NAMES,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support,
})


int8_metrics_df

,class_id,class_name,precision,recall,f1,support
0,0,OTHER,0.869198,0.903509,0.886022,228
1,1,SNORE,0.908537,0.897590,0.903030,166
2,2,COUGH,0.982917,0.973958,0.978417,768


In [19]:
int8_prediction_df = (
    test_df.copy()
)


int8_prediction_df[
    "predicted_class_id"
] = y_pred_int8


int8_prediction_df[
    "predicted_class_name"
] = [
    CLASS_NAMES[i]
    for i in y_pred_int8
]


int8_prediction_df[
    "confidence"
] = np.max(
    y_prob_int8,
    axis=1,
)


int8_prediction_df[
    "prob_other"
] = y_prob_int8[:, 0]

int8_prediction_df[
    "prob_snore"
] = y_prob_int8[:, 1]

int8_prediction_df[
    "prob_cough"
] = y_prob_int8[:, 2]


int8_prediction_df[
    "correct"
] = (
    y_true
    == y_pred_int8
)


print(
    int8_prediction_df[
        "correct"
    ].value_counts()
)

correct
True     1103
False      59
Name: count, dtype: int64


In [20]:
int8_dataset_metrics = (
    int8_prediction_df
    .groupby(
        "source_dataset"
    )
    .agg(
        samples=(
            "correct",
            "size",
        ),
        correct=(
            "correct",
            "sum",
        ),
        accuracy=(
            "correct",
            "mean",
        ),
    )
    .reset_index()
)


int8_dataset_metrics

,source_dataset,samples,correct,accuracy
0,AudioSet Strong,160,143,0.893750
1,Coswara,762,742,0.973753
2,ESC-50,240,218,0.908333


In [21]:
int8_stratum_metrics = (
    int8_prediction_df
    .groupby(
        [
            "source_dataset",
            "class_name",
        ]
    )
    .agg(
        samples=(
            "correct",
            "size",
        ),
        correct=(
            "correct",
            "sum",
        ),
        accuracy=(
            "correct",
            "mean",
        ),
    )
    .reset_index()
)


int8_stratum_metrics

,source_dataset,class_name,samples,correct,accuracy
0,AudioSet Strong,SNORE,160,143,0.893750
1,Coswara,COUGH,762,742,0.973753
2,ESC-50,COUGH,6,6,1.000000
3,ESC-50,OTHER,228,206,0.903509
4,ESC-50,SNORE,6,6,1.000000


In [24]:
float32_prediction_df = pd.read_csv(
    FLOAT32_PREDICTION_PATH
)


assert len(
    float32_prediction_df
) == len(
    int8_prediction_df
)


# TEST manifestの並びが同一か確認
if "clip_id" in test_df.columns:
    assert (
        float32_prediction_df[
            "clip_id"
        ].astype(str).to_numpy()
        ==
        int8_prediction_df[
            "clip_id"
        ].astype(str).to_numpy()
    ).all()


float32_pred = (
    float32_prediction_df[
        "predicted_class_id"
    ]
    .astype(np.int32)
    .to_numpy()
)


float32_correct = (
    float32_prediction_df[
        "correct"
    ]
    .astype(bool)
    .to_numpy()
)


int8_correct = (
    y_true
    == y_pred_int8
)


prediction_changed = (
    float32_pred
    != y_pred_int8
)


print(
    "Prediction changed:",
    prediction_changed.sum()
)


print(
    "float32 correct -> int8 correct:",
    np.sum(
        float32_correct
        & int8_correct
    )
)

print(
    "float32 correct -> int8 wrong:",
    np.sum(
        float32_correct
        & ~int8_correct
    )
)

print(
    "float32 wrong -> int8 correct:",
    np.sum(
        ~float32_correct
        & int8_correct
    )
)

print(
    "float32 wrong -> int8 wrong:",
    np.sum(
        ~float32_correct
        & ~int8_correct
    )
)

Prediction changed: 5
float32 correct -> int8 correct: 1101
float32 correct -> int8 wrong: 3
float32 wrong -> int8 correct: 2
float32 wrong -> int8 wrong: 56


In [26]:
FLOAT32_ACCURACY = (
    0.9500860571861267
)

FLOAT32_MACRO_F1 = (
    0.9234110352752863
)


accuracy_diff = (
    int8_accuracy
    - FLOAT32_ACCURACY
)

macro_f1_diff = (
    int8_macro_f1
    - FLOAT32_MACRO_F1
)


comparison_df = pd.DataFrame({
    "model": [
        "float32",
        "int8_ptq",
    ],
    "accuracy": [
        FLOAT32_ACCURACY,
        int8_accuracy,
    ],
    "macro_f1": [
        FLOAT32_MACRO_F1,
        int8_macro_f1,
    ],
})


print(
    comparison_df
)

print()

print(
    "Accuracy difference:",
    accuracy_diff
)

print(
    "Macro-F1 difference:",
    macro_f1_diff
)

print(
    "Accuracy change:",
    accuracy_diff * 100,
    "percentage points"
)

print(
    "Macro-F1 change:",
    macro_f1_diff
)

      model  accuracy  macro_f1
0   float32  0.950086  0.923411
1  int8_ptq  0.949225  0.922490

Accuracy difference: -0.0008605838642677943
Macro-F1 difference: -0.0009213437440538375
Accuracy change: -0.08605838642677943 percentage points
Macro-F1 change: -0.0009213437440538375


In [27]:
ACCURACY_DROP_LIMIT = 0.01
MACRO_F1_DROP_LIMIT = 0.01


accuracy_drop = (
    FLOAT32_ACCURACY
    - int8_accuracy
)

macro_f1_drop = (
    FLOAT32_MACRO_F1
    - int8_macro_f1
)


ptq_accuracy_ok = (
    accuracy_drop
    <= ACCURACY_DROP_LIMIT
)

ptq_macro_f1_ok = (
    macro_f1_drop
    <= MACRO_F1_DROP_LIMIT
)


ptq_acceptable = (
    ptq_accuracy_ok
    and ptq_macro_f1_ok
)


print(
    "Accuracy drop:",
    accuracy_drop
)

print(
    "Macro-F1 drop:",
    macro_f1_drop
)

print()

print(
    "Accuracy criterion:",
    ptq_accuracy_ok
)

print(
    "Macro-F1 criterion:",
    ptq_macro_f1_ok
)

print()

print(
    "PTQ acceptable:",
    ptq_acceptable
)

Accuracy drop: 0.0008605838642677943
Macro-F1 drop: 0.0009213437440538375

Accuracy criterion: True
Macro-F1 criterion: True

PTQ acceptable: True


In [28]:
quantization_metrics_df = pd.DataFrame([
    {
        "quantization_method":
            "full_integer_int8_ptq",

        "representative_split":
            "TRAIN",

        "representative_samples":
            len(train_df),

        "input_dtype":
            str(
                input_info["dtype"]
            ),

        "input_scale":
            float(
                input_scale
            ),

        "input_zero_point":
            int(
                input_zero_point
            ),

        "output_dtype":
            str(
                output_info["dtype"]
            ),

        "output_scale":
            float(
                output_scale
            ),

        "output_zero_point":
            int(
                output_zero_point
            ),

        "float32_keras_size_bytes":
            float32_file_size,

        "int8_tflite_size_bytes":
            int8_file_size,

        "float32_accuracy":
            FLOAT32_ACCURACY,

        "int8_accuracy":
            int8_accuracy,

        "accuracy_difference":
            accuracy_diff,

        "float32_macro_f1":
            FLOAT32_MACRO_F1,

        "int8_macro_f1":
            int8_macro_f1,

        "macro_f1_difference":
            macro_f1_diff,

        "prediction_changed_count":
            int(
                prediction_changed.sum()
            ),

        "float32_correct_int8_wrong":
            int(
                np.sum(
                    float32_correct
                    & ~int8_correct
                )
            ),

        "float32_wrong_int8_correct":
            int(
                np.sum(
                    ~float32_correct
                    & int8_correct
                )
            ),

        "ptq_acceptable":
            bool(
                ptq_acceptable
            ),
    }
])


quantization_metrics_df.T

,0
quantization_method,full_integer_int8_ptq
representative_split,TRAIN
representative_samples,5425
input_dtype,<class 'numpy.int8'>
input_scale,0.021448
input_zero_point,2
output_dtype,<class 'numpy.int8'>
output_scale,0.003906
output_zero_point,-128
float32_keras_size_bytes,132633


In [29]:
quantization_metrics_df.to_csv(
    AI08_META
    / "quantization_metrics.csv",
    index=False,
)


int8_metrics_df.to_csv(
    AI08_META
    / "int8_classification_metrics.csv",
    index=False,
)


pd.DataFrame(
    cm_int8,
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
).to_csv(
    AI08_META
    / "int8_confusion_matrix.csv"
)


int8_prediction_df.to_csv(
    AI08_META
    / "int8_test_predictions.csv",
    index=False,
)


int8_dataset_metrics.to_csv(
    AI08_META
    / "int8_dataset_metrics.csv",
    index=False,
)


int8_stratum_metrics.to_csv(
    AI08_META
    / "int8_stratum_metrics.csv",
    index=False,
)


print(
    "Saved AI-08 metadata to:",
    AI08_META
)

Saved AI-08 metadata to: /content/drive/MyDrive/TRON/AI/dataset/metadata/08_quant_info
